In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 4.1 Systematic Model Comparison
## Logistic Regression vs. Random Forest vs. XGBoost
- Three most practical models, head-to-head
- Interpretability vs. performance trade-off
- Making a justified recommendation

## Setup

In [ ]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.metrics import (precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score,
    confusion_matrix, brier_score_loss, classification_report)

RANDOM_STATE = 42

train_df = pd.read_csv('../data/training.csv')
test_df = pd.read_csv('../data/testing.csv')
train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',
    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']

# Raw features for the logistic PIPELINE (it encodes + scales internally)
X_test_raw = test_df[numeric_features + categorical_features].copy()

# Encoded matrix for the TREE models (manual dummies, aligned to training columns)
train_enc = pd.get_dummies(train_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
test_enc = pd.get_dummies(test_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)

# Impute with TRAIN medians only, never test's own (avoids leakage)
train_medians = train_enc.median()
test_enc = test_enc.fillna(train_medians)

X_test, y_test = test_enc, test_df['DEPARTED']

print(f"Testing: {X_test.shape[0]:,} | Encoded features: {X_test.shape[1]} | Raw cols: {X_test_raw.shape[1]}")
print(f"Departure rate: {y_test.mean():.2%} (test)")

## Load the Tuned Models

In [ ]:
lr = joblib.load('../models/best_tuned_logistic_model.pkl')
rf = joblib.load('../models/rf_tuned_f1.pkl')
xgb = joblib.load('../models/xgb_tuned_f1.pkl')

# Logistic Regression: pipeline trained on raw SEM_3_STATUS labels ('E'/'N')
lr_pred = (lr.predict(X_test_raw) != 'E').astype(int)
lr_prob = lr.predict_proba(X_test_raw)[:, 1]

# Random Forest
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

# XGBoost
xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)[:, 1]

print("Loaded tuned models and generated predictions.")

## Performance Comparison

In [ ]:
model_results = []
models = {
    'Regularized Logistic': {'preds': lr_pred, 'probs': lr_prob},
    'Random Forest': {'preds': rf_pred, 'probs': rf_prob},
    'XGBoost': {'preds': xgb_pred, 'probs': xgb_prob}
}

for name, d in models.items():
    model_results.append({
        'Model': name,
        'Precision': precision_score(y_test, d['preds']),
        'Recall': recall_score(y_test, d['preds']),
        'F1 Score': f1_score(y_test, d['preds']),
        'Avg Precision': average_precision_score(y_test, d['probs']),
        'Brier Score': brier_score_loss(y_test, d['probs'])
    })

results_df = pd.DataFrame(model_results).set_index('Model')
print("=" * 90)
print("HEAD-TO-HEAD MODEL COMPARISON")
print("=" * 90)
print(results_df.to_string(float_format="%.4f"))

## Precision-Recall Curves

In [ ]:
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
fig = go.Figure()

for i, (name, prob) in enumerate([('Regularized Logistic', lr_prob),
                                    ('Random Forest', rf_prob), ('XGBoost', xgb_prob)]):
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    fig.add_trace(go.Scatter(x=rec, y=prec, mode='lines',
        name=f'{name} (AP={ap:.3f})', line=dict(color=colors[i], width=2)))

baseline = y_test.mean()
fig.add_trace(go.Scatter(x=[0, 1], y=[baseline, baseline], mode='lines',
    name=f'No-skill ({baseline:.3f})', line=dict(color='gray', dash='dash')))

fig.update_layout(height=450, title_text='Precision-Recall Curves')
fig.update_xaxes(title_text='Recall')
fig.update_yaxes(title_text='Precision')
fig.show()

## Interpretability vs. Performance

In [ ]:
max_f1 = results_df['F1 Score'].max()
f1_lr = (results_df.loc['Regularized Logistic', 'F1 Score'] / max_f1) * 10
f1_rf = (results_df.loc['Random Forest', 'F1 Score'] / max_f1) * 10
f1_xgb = (results_df.loc['XGBoost', 'F1 Score'] / max_f1) * 10

dimensions = ['F1 (scaled to 10)', 'Interpretability', 'Training Speed',
              'Handles Non-linearity', 'Ease of Deployment']

data = {
    'Regularized Logistic': [f1_lr, 9, 9, 3, 9],
    'Random Forest': [f1_rf, 5, 7, 8, 7],
    'XGBoost': [f1_xgb, 3, 6, 9, 6]
}

df_radar = pd.DataFrame(data, index=dimensions)

fig = go.Figure()
for col in df_radar.columns:
    fig.add_trace(go.Scatterpolar(
        r=df_radar[col].values, theta=df_radar.index,
        fill='toself', name=col, opacity=0.3, line_width=2
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 10])),
    showlegend=True,
    title_text='Model Comparison: Performance vs. Practicality'
)
fig.show()

## Recommendations for Higher Education

### Decision Framework

| Your Priority | Recommended Model | Why |
|:-------------|:-----------------|:----|
| **Explainability to advisors** | Regularized Logistic Regression | Coefficients show clear factor contributions |
| **Reliable risk scoring** | Random Forest | Robust, handles messy data well |
| **Maximum predictive accuracy** | XGBoost | Typically highest AUC and F1 |
| **Research publications** | XGBoost | Best metrics for academic papers |

### Practical Recommendation

For most higher education institutions, we recommend a **two-model approach**:

1. **Regularized Logistic Regression** for stakeholder-facing outputs (reports, advisor dashboards, compliance)
2. **Random Forest or XGBoost** for backend risk scoring where performance matters most

## Summary

### Key Findings

1. All three models are strong performers on student departure prediction
2. The performance gap between models is often smaller than expected — interpretability and deployment considerations often matter more
3. Regularized Logistic Regression remains highly competitive while being fully transparent
4. XGBoost typically edges ahead on raw metrics but requires more infrastructure
5. Random Forest provides an excellent middle ground

**Next:** 4.2 Final Model Selection and Deployment